In [47]:
# setting up / configuring 
import sys
assert sys.version_info >= (3, 5)

# Scikit-Learn ≥0.20 is required
import sklearn
assert sklearn.__version__ >= "0.20"

# setting the plot dpi 
#plt.rcParams['figure.dpi']= 1000

# Common imports
import numpy as np
import os

# pandas
import pandas as pd

# to make this notebook's output stable across runs
np.random.seed(42)

# To plot pretty figures
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)

# Where to save the figures
PROJECT_ROOT_DIR = "."
CHAPTER_ID = "training_linear_models"
IMAGES_PATH = os.path.join(PROJECT_ROOT_DIR, "images", CHAPTER_ID)
os.makedirs(IMAGES_PATH, exist_ok=True)

def save_fig(fig_id, tight_layout=True, fig_extension="png", resolution=300):
    path = os.path.join(IMAGES_PATH, fig_id + "." + fig_extension)
    print("Saving figure", fig_id)
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)

In [48]:
# reading the csv file and printing the head 
df = pd.read_csv("C:/Users/andsa/Documents/ML/concrete_data.csv")  
df.head()

# removing outliers, cleaning the data
from scipy import stats

# filtering out the non numeric values 
#df_numerical = df.select_dtypes(include = ['float64', 'int64'])

z_scores = stats.zscore(df)
# threshold of 3 std deviations 
threshold = 3
# using a bool mask to remove outliers 
mask = (z_scores < threshold).all(axis = 1)

df_cleaned = df[mask]

# Checking for correlation 
#corr_matrix = df.corr()
#corr_matrix["Strength"]

# looking as a histogram of the params 
#df.hist()

In [49]:
# assigning the X and y values 
X = df[['Cement', 'Blast Furnace Slag', 'Fly Ash', 'Water', 'Superplasticizer', 'Coarse Aggregate', 'Fine Aggregate', 'Age']]
y = df[['Strength']]

In [50]:
# importing test metrics 
from sklearn.metrics import mean_squared_error, r2_score

# importing a splitter for the data 
from sklearn.model_selection import train_test_split

# Split into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scaling the data - think this won't allow the same pairs in test and train, scaling is important for the SGD regressor 
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# checking for skewness of the strength distribution 
#from scipy.stats import skew
#skewness = skew(df['Strength'])
#print(skewness)




In [51]:
# Doing an ordinary least squares regression -> OLS 
from sklearn.linear_model import LinearRegression
lin_reg_OLS = LinearRegression()

# fitting the OLS model to the training data
lin_reg_OLS.fit(X_train, y_train)

# testing out the lasso method
from sklearn.linear_model import Lasso
lin_reg_Lasso = Lasso(alpha=0.1)  

# fitting the model to the training data
lin_reg_Lasso.fit(X_train_scaled, y_train)



# Doing a stochastic gradient descent method -> SGD
from sklearn.linear_model import SGDRegressor
#lin_reg_SGD = SGDRegressor(max_iter=1300, eta0=0.01, learning_rate="constant")
#lin_reg_SGD = SGDRegressor(max_iter = 4000, learning_rate= "optimal", random_state = 42)
lin_reg_SGD = SGDRegressor(max_iter=5000, tol=1e-5, eta0=0.01, learning_rate='constant', random_state=42)

# fitting the SGD model to the training data 
lin_reg_SGD.fit(X_train_scaled, y_train)

c:\Users\andsa\anaconda3\lib\site-packages\sklearn\utils\validation.py:63: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  return f(*args, **kwargs)


SGDRegressor(learning_rate='constant', max_iter=5000, random_state=42,
             tol=1e-05)

In [52]:
# Predicting strength values on the pred data set 

y_pred_OLS = lin_reg_OLS.predict(X_test)
y_pred_SGD = lin_reg_SGD.predict(X_test_scaled)
y_pred_Lasso = lin_reg_Lasso.predict(X_test_scaled)



# printing the metrics for the OLS model 
print("OLS Regression:")
print(f"R² Score: {np.round(r2_score(y_test, y_pred_OLS), 4)}")
print(f"MSE: {np.round(mean_squared_error(y_test, y_pred_OLS), 2)}\n")

# printing the metrics for the Lasso model 
print("Lasso Regression:")
print(f"R² Score: {np.round(r2_score(y_test, y_pred_Lasso), 4)}")
print(f"MSE: {np.round(mean_squared_error(y_test, y_pred_OLS), 2)}\n")

# printing the metrics for the SGD model 
print("Mini-batch SGD Regression:")
print(f"R² Score: {np.round(r2_score(y_test, y_pred_SGD), 2)}")
print(f"MSE: {np.round(mean_squared_error(y_test, y_pred_SGD), 2)}\n")



# doing cross validation
from sklearn.model_selection import cross_val_score

# Perform 5-fold cross-validation using R-squared for estimating the performance 
cv_scores = cross_val_score(lin_reg_SGD, X_train_scaled, y_train, cv=5, scoring='r2')

# Print the average score across folds
print(f'Average R-squared score from 5-fold cross-validation: {np.round(cv_scores.mean(), 4)}\n')


OLS Regression:
R² Score: 0.6276
MSE: 95.97

Lasso Regression:
R² Score: 0.6258
MSE: 95.97

Mini-batch SGD Regression:
R² Score: 0.63
MSE: 95.27

Average R-squared score from 5-fold cross-validation: 0.5665



c:\Users\andsa\anaconda3\lib\site-packages\sklearn\utils\validation.py:63: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  return f(*args, **kwargs)
c:\Users\andsa\anaconda3\lib\site-packages\sklearn\utils\validation.py:63: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  return f(*args, **kwargs)
c:\Users\andsa\anaconda3\lib\site-packages\sklearn\utils\validation.py:63: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  return f(*args, **kwargs)
c:\Users\andsa\anaconda3\lib\site-packages\sklearn\utils\validation.py:63: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using r

In [53]:
# importing the grid search feature to optimize the params for SGD
from sklearn.model_selection import GridSearchCV

# Define hyperparameter grid with some values
params_for_grid = {
    "eta0": [0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05, 0.1],  # More values
    "max_iter": [500, 1000, 1500, 2000, 3000, 5000],  # Increased range
    "learning_rate": ["constant", "optimal", "invscaling", "adaptive"]
}

# Creating the SGD regressor
SGD = SGDRegressor(random_state=42)

# Perform Grid Search, dividing into 5 folds -> cv = 5, using all CPU cores to speed it up -> n_jobs = -1 
grid_search = GridSearchCV(SGD, params_for_grid, cv=5, scoring="r2", n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)

# Getting the best model from the grid search
best_SGD = grid_search.best_estimator_

# Predicting on the test set
y_pred = best_SGD.predict(X_test_scaled)

# Computing MSE
mse = mean_squared_error(y_test, y_pred)

# Printing the results for the "optimal" SGD model and it's parameters 
print("Best Parameters:", grid_search.best_params_)
print("Best R² Score (CV):", grid_search.best_score_)
print("Test MSE:", mse)



Best Parameters: {'eta0': 0.1, 'learning_rate': 'adaptive', 'max_iter': 500}
Best R² Score (CV): 0.5945867371010949
Test MSE: 95.86890494034817


c:\Users\andsa\anaconda3\lib\site-packages\sklearn\utils\validation.py:63: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  return f(*args, **kwargs)


In [67]:
# polynomial regression 
from sklearn.preprocessing import PolynomialFeatures


poly_features = PolynomialFeatures(degree = 3)
X_poly_train = poly_features.fit_transform(X_train_scaled)
X_poly_test = poly_features.transform(X_test_scaled)


lin_reg2 = LinearRegression()
lin_reg2.fit(X_poly_train, y_train)

y_pred_poly2 = lin_reg2.predict(X_poly_test)

print("Simple poly Regression:")
print(f"R² Score: {np.round(r2_score(y_test, y_pred_poly2), 4)}")
print(f"MSE: {np.round(mean_squared_error(y_test, y_pred_poly2), 2)}\n")


Simple poly Regression:
R² Score: 0.8436
MSE: 40.31



In [92]:
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score

# Create a pipeline with PolynomialFeatures, StandardScaler, and Lasso regression
pipeline = Pipeline([
    ('poly', PolynomialFeatures()),    # Polynomial features
    ('scaler', StandardScaler()),      # Scaling the features
    ('regressor', Lasso())             # Lasso regression
])

# Define the parameter grid for GridSearchCV
param_grid = {
    'poly__degree': [1, 2, 3, 4, 5],       # Degrees of the polynomial
    'regressor__alpha': [0.1, 0.5, 1.0, 10],  # Regularization strength for Lasso
    'regressor__fit_intercept': [True, False]  # Whether or not to fit the intercept term
}

# Perform Grid Search with 5-fold cross-validation
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='neg_mean_squared_error')

# Fit the model using the grid search
grid_search.fit(X_train, y_train)

# Get the best parameters from the grid search
best_params = grid_search.best_params_
print(f"Best Parameters: {best_params}")

# Predict using the best model
y_pred_lasso_grid = grid_search.predict(X_test)

# Evaluate the model
print(f"R² Score: {np.round(r2_score(y_test, y_pred_lasso_grid), 4)}")
print(f"MSE: {np.round(mean_squared_error(y_test, y_pred_lasso_grid), 4)}")


c:\Users\andsa\anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:530: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 896.7953997084405, tolerance: 112.04855503740572
  model = cd_fast.enet_coordinate_descent(
c:\Users\andsa\anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:530: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 729.6899490518263, tolerance: 112.04855503740572
  model = cd_fast.enet_coordinate_descent(


Best Parameters: {'poly__degree': 2, 'regressor__alpha': 0.1, 'regressor__fit_intercept': True}
R² Score: 0.9984
MSE: 15.8236


c:\Users\andsa\anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:530: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 708.1238294650102, tolerance: 112.04855503740572
  model = cd_fast.enet_coordinate_descent(


In [93]:
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score

# Create a pipeline with PolynomialFeatures, StandardScaler, and RandomForestRegressor
pipeline = Pipeline([
    ('poly', PolynomialFeatures()),    # Polynomial features
    ('scaler', StandardScaler()),      # Scaling the features
    ('regressor', RandomForestRegressor())  # Random Forest regressor
])

# Define the parameter grid for GridSearchCV
param_grid = {
    'poly__degree': [1, 2, 3, 4, 5],          # Degrees of the polynomial
    'regressor__n_estimators': [50, 100, 200],  # Number of trees in the forest
    'regressor__max_depth': [None, 10, 20, 30],  # Maximum depth of the trees
    'regressor__min_samples_split': [2, 5, 10],  # Minimum number of samples required to split a node
    'regressor__min_samples_leaf': [1, 2, 4]     # Minimum number of samples required to be at a leaf node
}

# Perform Grid Search with 5-fold cross-validation
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='neg_mean_squared_error')

# Fit the model using the grid search
grid_search.fit(X_train, y_train)

# Get the best parameters from the grid search
best_params = grid_search.best_params_
print(f"Best Parameters: {best_params}")

# Predict using the best model
y_pred_rf_grid = grid_search.predict(X_test)

# Evaluate the model
print(f"R² Score: {np.round(r2_score(y_test, y_pred_rf_grid), 4)}")
print(f"MSE: {np.round(mean_squared_error(y_test, y_pred_rf_grid), 4)}")


c:\Users\andsa\anaconda3\lib\site-packages\sklearn\pipeline.py:346: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  self._final_estimator.fit(Xt, y, **fit_params_last_step)
c:\Users\andsa\anaconda3\lib\site-packages\sklearn\pipeline.py:346: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  self._final_estimator.fit(Xt, y, **fit_params_last_step)
c:\Users\andsa\anaconda3\lib\site-packages\sklearn\pipeline.py:346: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  self._final_estimator.fit(Xt, y, **fit_params_last_step)
c:\Users\andsa\anaconda3\lib\site-packages\sklearn\pipeline.py:346: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Plea

Best Parameters: {'poly__degree': 3, 'regressor__max_depth': None, 'regressor__min_samples_leaf': 1, 'regressor__min_samples_split': 2, 'regressor__n_estimators': 50}
R² Score: 0.9979
MSE: 19.893


c:\Users\andsa\anaconda3\lib\site-packages\sklearn\pipeline.py:346: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  self._final_estimator.fit(Xt, y, **fit_params_last_step)
